# Can we predict 30-day readmission?

This notebook evaluates whether the supplied encounter-level variables contain useful signal for `readmit_30d`. Because each person can have several visits, all visits from a person are kept in the same cross-validation fold. This estimates performance on **previously unseen people** and avoids patient leakage.

The analysis is exploratory, not a clinical validation. A useful model here would still require temporal/external validation, subgroup checks, calibration monitoring, and a defined intervention before deployment.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    log_loss, precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
pd.set_option('display.max_columns', 30)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load and understand the data

In [ ]:
DATA_PATH = Path('data.csv')
DICTIONARY_PATH = Path('columns.csv')

data = pd.read_csv(DATA_PATH)
dictionary = pd.read_csv(DICTIONARY_PATH)
display(dictionary)
display(data.head())
print(f'{len(data):,} visits from {data.person_key.nunique():,} people')

In [ ]:
target = 'readmit_30d'
required = set(dictionary['column'])
assert required == set(data.columns), 'Data columns do not match the dictionary.'
assert data[target].isin([0, 1]).all(), 'Target must be binary.'
assert not data.duplicated(['person_key', 'visit_number']).any(), 'Duplicate person/visit rows found.'

quality = pd.DataFrame({
    'dtype': data.dtypes.astype(str),
    'missing_n': data.isna().sum(),
    'missing_pct': data.isna().mean().mul(100).round(2),
    'unique_n': data.nunique(),
})
display(quality)

prevalence = data[target].mean()
visits_per_person = data.groupby('person_key').size()
print(f'Readmission prevalence: {prevalence:.1%} ({data[target].sum():,}/{len(data):,})')
print(f'Visits/person: median {visits_per_person.median():.0f}, range {visits_per_person.min()}–{visits_per_person.max()}')

### Leakage and feature timing

- `person_key` is an identifier, so it is used only to group folds and is excluded from the features.
- `postcode_area` is a category even though it is stored as an integer.
- `length_of_stay_days` and `n_medications` are available at discharge, so this is a **discharge-time** prediction. They would leak future information if the intended prediction time were admission.
- There is no encounter date. We therefore cannot simulate a prospective temporal split or derive age reliably from `birth_year`; the model uses birth year directly.

In [ ]:
feature_data = data.drop(columns=[target, 'person_key']).copy()
feature_data['postcode_area'] = feature_data['postcode_area'].astype('string')
y = data[target].astype(int)
groups = data['person_key']

categorical_features = ['sex', 'postcode_area']
numeric_features = [c for c in feature_data.columns if c not in categorical_features]
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)

## 2. Grouped out-of-fold evaluation

We compare a prevalence-only baseline, regularized logistic regression, and nonlinear histogram gradient boosting. Every reported prediction is out-of-fold. Five-fold stratified group cross-validation balances outcomes approximately while never splitting a person across train and validation.

Primary discrimination metrics are ROC AUC and average precision (AP). AP is especially informative with the 19.8% positive class; the no-skill reference is approximately the prevalence. Brier score and log loss assess probability quality (lower is better).

In [ ]:
numeric_scaled = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_sparse = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
linear_preprocessor = ColumnTransformer([
    ('numeric', numeric_scaled, numeric_features),
    ('categorical', categorical_sparse, categorical_features),
])

tree_preprocessor = ColumnTransformer([
    ('numeric', SimpleImputer(strategy='median'), numeric_features),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), categorical_features),
])

models = {
    'Prevalence baseline': DummyClassifier(strategy='prior'),
    'Logistic regression': Pipeline([
        ('preprocess', linear_preprocessor),
        ('model', LogisticRegression(max_iter=2_000)),
    ]),
    'Histogram gradient boosting': Pipeline([
        ('preprocess', tree_preprocessor),
        ('model', HistGradientBoostingClassifier(
            max_iter=250, learning_rate=0.05, l2_regularization=2.0,
            random_state=RANDOM_STATE,
        )),
    ]),
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_predictions = {}
for name, model in models.items():
    oof_predictions[name] = cross_val_predict(
        model, feature_data, y, groups=groups, cv=cv,
        method='predict_proba', n_jobs=-1,
    )[:, 1]

def score_predictions(y_true, probability):
    return {
        'ROC AUC': roc_auc_score(y_true, probability),
        'Average precision': average_precision_score(y_true, probability),
        'Brier score': brier_score_loss(y_true, probability),
        'Log loss': log_loss(y_true, probability),
    }

results = pd.DataFrame({
    name: score_predictions(y, probability)
    for name, probability in oof_predictions.items()
}).T.sort_values('ROC AUC', ascending=False)
display(results.style.format('{:.3f}'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
colors = {'Prevalence baseline': '#777777', 'Logistic regression': '#176B87',
          'Histogram gradient boosting': '#C65D3B'}

for name, probability in oof_predictions.items():
    fpr, tpr, _ = roc_curve(y, probability)
    precision, recall, _ = precision_recall_curve(y, probability)
    observed, predicted = calibration_curve(y, probability, n_bins=10, strategy='quantile')
    axes[0].plot(fpr, tpr, label=f"{name} ({results.loc[name, 'ROC AUC']:.3f})", color=colors[name])
    axes[1].plot(recall, precision, label=f"{name} ({results.loc[name, 'Average precision']:.3f})", color=colors[name])
    axes[2].plot(predicted, observed, marker='o', label=name, color=colors[name])

axes[0].plot([0, 1], [0, 1], '--', color='black', alpha=.5)
axes[1].axhline(prevalence, linestyle='--', color='black', alpha=.5, label=f'Prevalence ({prevalence:.3f})')
axes[2].plot([0, 1], [0, 1], '--', color='black', alpha=.5)
axes[0].set(title='ROC curve', xlabel='False-positive rate', ylabel='True-positive rate')
axes[1].set(title='Precision–recall curve', xlabel='Recall', ylabel='Precision')
axes[2].set(title='Calibration (quantile bins)', xlabel='Mean predicted probability', ylabel='Observed frequency')
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Uncertainty for the best model

The same person contributes multiple rows, so an ordinary row bootstrap would understate uncertainty. The bootstrap below resamples people as clusters. These intervals describe sampling uncertainty in this dataset; they do not account for temporal or site shift.

In [ ]:
best_name = results.index[0]
best_probability = oof_predictions[best_name]

def clustered_bootstrap_metrics(y_true, probability, cluster, n_boot=1_000, seed=42):
    rng = np.random.default_rng(seed)
    unique_clusters = np.asarray(pd.unique(cluster))
    row_indices = {g: np.flatnonzero(np.asarray(cluster) == g) for g in unique_clusters}
    samples = []
    for _ in range(n_boot):
        sampled_clusters = rng.choice(unique_clusters, size=len(unique_clusters), replace=True)
        idx = np.concatenate([row_indices[g] for g in sampled_clusters])
        if np.unique(np.asarray(y_true)[idx]).size < 2:
            continue
        samples.append(score_predictions(np.asarray(y_true)[idx], np.asarray(probability)[idx]))
    return pd.DataFrame(samples)

bootstrap = clustered_bootstrap_metrics(y, best_probability, groups)
confidence_intervals = pd.DataFrame({
    'estimate': pd.Series(score_predictions(y, best_probability)),
    '95% CI lower': bootstrap.quantile(0.025),
    '95% CI upper': bootstrap.quantile(0.975),
})
print(f'Best model by grouped out-of-fold ROC AUC: {best_name}')
display(confidence_intervals.style.format('{:.3f}'))

## 4. Illustrative operating points

A probability model does not determine whom to flag; that depends on the cost and capacity of the intervention. The table shows several **exploratory** thresholds using the out-of-fold predictions. Selecting a threshold from these data is optimistic, so a chosen threshold must be locked and tested on a new validation set.

In [ ]:
def operating_point(y_true, probability, threshold):
    prediction = probability >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        'Threshold': threshold,
        'Flagged (%)': prediction.mean() * 100,
        'Sensitivity': tp / (tp + fn),
        'Specificity': tn / (tn + fp),
        'Precision': tp / (tp + fp) if tp + fp else np.nan,
        'NPV': tn / (tn + fn) if tn + fn else np.nan,
    }

operating_points = pd.DataFrame([
    operating_point(y, best_probability, threshold)
    for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]
])
display(operating_points.style.format({
    'Threshold': '{:.2f}', 'Flagged (%)': '{:.1f}', 'Sensitivity': '{:.1%}',
    'Specificity': '{:.1%}', 'Precision': '{:.1%}', 'NPV': '{:.1%}',
}))

## 5. What drives the logistic model?

For a compact descriptive view, fit logistic regression to all available data and show its largest standardized coefficients. This is **not causal interpretation**. Correlated variables, area effects, and repeated observations can all affect coefficients; the fitted model below is for inspection and would need to be refit inside a final training protocol for use.

In [ ]:
logistic_model = models['Logistic regression'].fit(feature_data, y)
feature_names = logistic_model.named_steps['preprocess'].get_feature_names_out()
coefficients = pd.Series(
    logistic_model.named_steps['model'].coef_[0], index=feature_names, name='coefficient'
).sort_values()
largest = pd.concat([coefficients.head(10), coefficients.tail(10)]).sort_values()

fig, ax = plt.subplots(figsize=(8, 7))
colors_coef = np.where(largest >= 0, '#C65D3B', '#176B87')
largest.plot.barh(ax=ax, color=colors_coef)
ax.axvline(0, color='black', linewidth=0.8)
ax.set(title='Largest logistic-regression coefficients', xlabel='Log-odds coefficient')
plt.tight_layout()
plt.show()

## 6. Conclusion

The answer is **yes, but only weakly/modestly with these features**. Grouped out-of-fold evaluation should show logistic regression outperforming the prevalence baseline, while nonlinear boosting does not improve on it. The gain in average precision is meaningful relative to the roughly 20% event rate, but discrimination remains limited. This supports further investigation, not standalone clinical use.

Recommended next steps:

1. Add encounter timestamps and run a true temporal holdout; if deployment spans sites, add a site-level external holdout.
2. Confirm every feature is available at the exact prediction time.
3. Predefine the intervention and its capacity/cost, then choose a threshold accordingly.
4. Evaluate calibration and sensitivity across clinically relevant groups, especially sex, age, and geography; postcode may encode socioeconomic status and raises fairness/privacy concerns.
5. Compare against an existing clinical score or workflow, and assess whether predictions improve patient outcomes in a prospective study.

In [ ]:
auc = confidence_intervals.loc['ROC AUC']
ap = confidence_intervals.loc['Average precision']
print(
    f'Summary: {best_name} achieved grouped out-of-fold ROC AUC '    f"{auc['estimate']:.3f} (95% cluster-bootstrap CI {auc['95% CI lower']:.3f}–{auc['95% CI upper']:.3f}) "
    f"and average precision {ap['estimate']:.3f} ({ap['95% CI lower']:.3f}–{ap['95% CI upper']:.3f}), "
    f'against a positive prevalence of {prevalence:.3f}.'
)